In [1]:
import os
import sys

In [2]:
current_dir = os.getcwd()
root_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, root_dir)

In [3]:
from gitsource import GithubRepositoryDataReader

In [4]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

In [5]:
documents = [file.parse() for file in reader.read()]

In [6]:
len(documents)

72

In [7]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Generating ground truth

In [8]:
from pydantic import BaseModel

In [9]:
class Questions(BaseModel):
    questions: list[str]

In [10]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [3]:
import json

from tqdm.auto import tqdm
from dotenv import load_dotenv
from anthropic import Anthropic
from utils.evaluation_utils import llm_structured

In [4]:
load_dotenv()

True

In [14]:
anthropic_client = Anthropic()

In [18]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured(
        anthropic_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })
    return results, usage

## Generating Questions

### For the first three (3) documents

In [33]:
[doc['filename'] for doc in documents[:3]] 

['01-agentic-rag/lessons/01-intro.md',
 '01-agentic-rag/lessons/02-environment.md',
 '01-agentic-rag/lessons/03-rag.md']

In [34]:
ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.append(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [35]:
usages

[Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1300, output_tokens=103, output_tokens_details=None, server_tool_use=None, service_tier='standard'),
 Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1688, output_tokens=109, output_tokens_details=None, server_tool_use=None, service_tier='standard'),
 Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=2197, output_tokens=119, output_tokens_details=None, server_tool_use=None, service_tier='standard')]

#### Q1. Average number of input tokens across these 3 calls

In [45]:
avg_num_input_tok = sum([usage.input_tokens for usage in usages]) / len(usages)

In [46]:
avg_num_input_tok

1728.3333333333333

In [36]:
from utils.evaluation_utils import calc_total_price

In [37]:
calc_total_price(usages)

0.00684

### Apply across all documents

In [22]:
from concurrent.futures import ThreadPoolExecutor
from utils.evaluation_utils import map_progress

In [23]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/72 [00:00<?, ?it/s]

In [25]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

360

In [26]:
calc_total_price(usages)

0.17667200000000002

In [8]:
import pandas as pd

In [29]:
df_ground_truth = pd.DataFrame(ground_truth)

In [30]:
df_ground_truth.head()

,question,document
0,What is a Large Language Model and how does it...,01-agentic-rag/lessons/01-intro.md
1,What are the main limitations that LLMs face w...,01-agentic-rag/lessons/01-intro.md
2,How does RAG address the problems with LLMs no...,01-agentic-rag/lessons/01-intro.md
3,What will be built in this module and what pro...,01-agentic-rag/lessons/01-intro.md
4,What are the main differences between Part 1 a...,01-agentic-rag/lessons/01-intro.md


In [31]:
df_ground_truth.to_csv('data/ground-truth-new.csv', index=False)

## Loading the ground truth

In [9]:
df_ground_truth = pd.read_csv('data/ground-truth-new.csv')

In [10]:
df_ground_truth = df_ground_truth.rename(columns={"document": "filename"})

In [11]:
df_ground_truth.head()

,question,filename
0,What is a Large Language Model and how does it...,01-agentic-rag/lessons/01-intro.md
1,What are the main limitations that LLMs face w...,01-agentic-rag/lessons/01-intro.md
2,How does RAG address the problems with LLMs no...,01-agentic-rag/lessons/01-intro.md
3,What will be built in this module and what pro...,01-agentic-rag/lessons/01-intro.md
4,What are the main differences between Part 1 a...,01-agentic-rag/lessons/01-intro.md


In [12]:
ground_truth = df_ground_truth.to_dict(orient="records")

## Searching the chunks

In [13]:
from gitsource import chunk_documents

In [14]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
len(chunks)

295

In [16]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [17]:
from minsearch import Index, VectorSearch
from embed.embedder import Embedder

import numpy as np

In [18]:
model = Embedder("../embed/models/Xenova/all-MiniLM-L6-v2/")

../embed/models/Xenova/all-MiniLM-L6-v2


In [38]:
def build_index(documents):
    index = Index(    
        text_fields = ['content'],
        keyword_fields = ['filename']
    )
    index.fit(documents)
    
    return index

def build_vindex(vectors, documents):
    vindex = VectorSearch(
        keyword_fields=['filename']
    )
    vindex.fit(vectors, documents)

    return vindex

In [39]:
index = build_index(chunks)

In [21]:
X = np.array([model.encode(chunk['content']) for chunk in chunks])
vindex = build_vindex(X, chunks)

In [22]:
def text_search(query, num_results=5):
    return index.search(
        query=query,
        num_results=num_results
    )

def vector_search(query, num_results=5):
    query_vector = model.encode(query)
    
    return vindex.search(
        query_vector,
        num_results=num_results
    )

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [23]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

### Q2. First result with text search

In [24]:
q = ground_truth[0]['question']

In [25]:
print(q)

What is a Large Language Model and how does it work at a basic level?


In [26]:
text_search_results = text_search(q)

In [27]:
text_search_results[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

### Q3. First result with vector search

In [28]:
vector_search_results = vector_search(q)

In [29]:
vector_search_results[0]

{'start': 1000,
 'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they sometim

## Evaluation metrics

In [30]:
def compute_relevance(q, search_function):
    doc_filename = q['filename']
    results = search_function(query=q['question'])

    return [1 if doc['filename'] == doc_filename else 0 for doc in results]

In [31]:
def compute_relevance_total(gr_tr, search_function):
    relevance_total = []
    for q in tqdm(gr_tr):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)
    return relevance_total

In [32]:
def hit_rate(relevance_matrix):
    count = 0
    for d in relevance_matrix:
        if 1 in d:
            count += 1
    return count / len(relevance_matrix)

In [33]:
def mrr(relevance_matrix):
    total_score = 0
    
    for row in relevance_matrix:
        for rank in range(len(row)):
            if row[rank] == 1:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance_matrix)

In [34]:
def evaluate(gr_tr, search_function):
    relevance_total = compute_relevance_total(gr_tr, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total)
    }

### Q4. Evaluating text search

In [36]:
from tqdm.auto import tqdm

In [40]:
evaluate(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8, 'mrr': 0.6495370370370372}

### Q5. Evaluating vector search

In [41]:
evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7472222222222222, 'mrr': 0.5645370370370373}

### Q6. Evaluating hybrid search

In [42]:
evaluate(ground_truth, lambda query, k=1: hybrid_search(query, k))

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8694444444444445, 'mrr': 0.6946759259259263}

In [43]:
evaluate(ground_truth, lambda query, k=50: hybrid_search(query, k))

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8527777777777777, 'mrr': 0.6781481481481484}

In [44]:
evaluate(ground_truth, lambda query, k=100: hybrid_search(query, k))

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8527777777777777, 'mrr': 0.6781481481481484}

In [45]:
evaluate(ground_truth, lambda query, k=200: hybrid_search(query, k))

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8527777777777777, 'mrr': 0.6781481481481484}

## Using Tuning for keyword search

### With GridSearch

In [47]:
def search_boosts(query, content_boost, filename_boost):
    boost_dict = {
        "content": content_boost,
        "filename": filename_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [48]:
results = []

for filename_boost in [1.0, 2.0, 5.0]:
    for content_boost in [1.0, 2.0, 4.0, 10.0]:
        print(
            f"Evaluating content_boost={content_boost},"
            f" filename_boost={filename_boost},"
        )
        result = evaluate(
            ground_truth,
            lambda query, content_boost=content_boost, filename_boost=filename_boost: search_boosts(
                query,
                content_boost,
                filename_boost,
            )
        )

        results.append({
            "content": content_boost,
            "filename": filename_boost,
            "hit_rate": result["hit_rate"],
            "mrr": result["mrr"],
        })

Evaluating content_boost=1.0, filename_boost=1.0,


  0%|          | 0/360 [00:00<?, ?it/s]

Evaluating content_boost=2.0, filename_boost=1.0,


  0%|          | 0/360 [00:00<?, ?it/s]

Evaluating content_boost=4.0, filename_boost=1.0,


...

  0%|          | 0/360 [00:00<?, ?it/s]

Evaluating content_boost=4.0, filename_boost=5.0,


  0%|          | 0/360 [00:00<?, ?it/s]

Evaluating content_boost=10.0, filename_boost=5.0,


  0%|          | 0/360 [00:00<?, ?it/s]

In [49]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,content,filename,hit_rate,mrr
0,1.0,1.0,0.8,0.649537
1,2.0,1.0,0.8,0.649537
2,4.0,1.0,0.8,0.649537
3,10.0,1.0,0.8,0.649537
4,1.0,2.0,0.8,0.649537
5,2.0,2.0,0.8,0.649537
6,4.0,2.0,0.8,0.649537
7,10.0,2.0,0.8,0.649537
8,1.0,5.0,0.8,0.649537
9,2.0,5.0,0.8,0.649537


### With hyperopt

In [50]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

/home/day/Documents/Learning/llm-zoomcamp-code/.venv/lib/python3.14/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [52]:
# 1. Définir l'espace de recherche
space = {
    "boost_content":  hp.uniform("boost_content",  0.5, 5.0),
    "boost_filename": hp.uniform("boost_filename", 0.0, 3.0),
    "num_results":    hp.choice("num_results", [3, 5, 10, 20]),
}

In [53]:
# 2. Définir la fonction objectif
def objective(params):
    boost_dict = {
        "content":  params["boost_content"],
        "filename": params["boost_filename"],
    }
    k = params["num_results"]

    def search_fn(query):
        return index.search(query, boost_dict=boost_dict, num_results=k)

    scores = evaluate(ground_truth, search_fn)

    # hyperopt minimise → on retourne le négatif du hit_rate
    return {
        "loss": -scores["hit_rate"],
        "status": STATUS_OK,
        "mrr": scores["mrr"],
        "hit_rate": scores["hit_rate"],
        "params": params,
    }

In [54]:
# 3. Lancer l'optimisation
trials = Trials()

best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,      # TPE : smarter que random search
    max_evals=50,          # nombre d'itérations
    trials=trials,
)

print("Meilleurs paramètres trouvés :", best)

  0%|                                                                                                  | 0/50 [00:00<?, ?trial/s, best loss=?]

  0%|          | 0/360 [00:00<?, ?it/s]

  2%|█▋                                                                                    | 1/50 [00:00<00:21,  2.32trial/s, best loss: -0.8]

  0%|          | 0/360 [00:00<?, ?it/s]

  4%|███▍                                                                                  | 2/50 [00:00<00:19,  2.45trial/s, best loss: -0.8]

...

 96%|███████████████████████████████████████████████████████████████████▏  | 48/50 [00:19<00:00,  2.53trial/s, best loss: -0.9305555555555556]

  0%|          | 0/360 [00:00<?, ?it/s]

 98%|████████████████████████████████████████████████████████████████████▌ | 49/50 [00:19<00:00,  2.54trial/s, best loss: -0.9305555555555556]

  0%|          | 0/360 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████| 50/50 [00:19<00:00,  2.53trial/s, best loss: -0.9305555555555556]
Meilleurs paramètres trouvés : {'boost_content': np.float64(3.5315439702568994), 'boost_filename': np.float64(0.08183546250686735), 'num_results': np.int64(3)}


In [55]:
def search_boosts_with_hyperopt(query):
    boost_dict = {
        "content": 3.5315439702568994,
        "filename": 0.08183546250686735,
    }

    return index.search(
        query,
        num_results=3,
        boost_dict=boost_dict,
    )

In [56]:
evaluate(ground_truth, search_boosts_with_hyperopt)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7388888888888889, 'mrr': 0.6356481481481484}